# Fraud Detection - Vertex AI Experiments Tracker

Trains two LightGBM models and logs **every metric, parameter, confusion matrix, and ROC curve** to Vertex AI Experiments so you can compare them side-by-side in the Vertex AI console.

**Where to see results after running:**
Vertex AI > Experiments > `fraud-detection-neo4j` > select both runs > Compare

In [1]:
import sys, subprocess

# Fix NumPy/PyArrow conflict common in Colab Enterprise
# (newer PyArrow in ~/.local clashes with conda NumPy 1.x)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'numpy<2',
    'pyarrow>=12,<15',
    'scikit-learn>=1.3',
    'lightgbm>=4.0',
    'gcsfs',
    'google-cloud-aiplatform',
], check=True)

print("Packages ready. Restarting kernel to apply changes...")

# Kernel will restart - re-run all cells after this
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

zsh:1: command not found: pip


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import subprocess
PROJECT_ID      = subprocess.run(['gcloud','config','get-value','project'],
                                  capture_output=True, text=True).stdout.strip()
LOCATION        = 'us-east1'
EXPERIMENT_NAME = 'fraud-detection-neo4j'
GCS_BUCKET      = 'neo4j-fraud-detection'
DATASET_PATH    = f'gs://{GCS_BUCKET}/artifacts/demo_dataset.parquet'
ARTIFACTS_PATH  = f'gs://{GCS_BUCKET}/experiment-artifacts'

print(f'Project  : {PROJECT_ID}')
print(f'Location : {LOCATION}')
print(f'Experiment: {EXPERIMENT_NAME}')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json, os
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.decomposition import PCA
import lightgbm as lgb
from matplotlib.patches import Patch
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, experiment=EXPERIMENT_NAME)
print('Vertex AI Experiments initialised.')
print(f'View at: https://console.cloud.google.com/vertex-ai/experiments?project={PROJECT_ID}')

---
## 1. Load Dataset

In [ ]:
print(f'Loading {DATASET_PATH} ...')
df = pd.read_parquet(DATASET_PATH)
print(f'Shape : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Fraud : {df["isFraud"].sum():,} ({df["isFraud"].mean():.2%})')

CAT_COLS = ['card4','card6','ProductCD','P_emaildomain','R_emaildomain']
for col in CAT_COLS:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str).fillna('nan'))

# M1-M9 are T/F/NaN match flags - map to 1/0/NaN (LightGBM handles NaN natively)
for col in [c for c in df.columns if c.startswith('M') and df[c].dtype == object]:
    df[col] = df[col].map({'T': 1, 'F': 0})

# Catch-all: encode any remaining object columns (safety net)
for col in df.select_dtypes(include='object').columns:
    if col not in {'isFraud', 'TransactionID', 'TransactionDT'}:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str).fillna('nan'))

GRAPH_COLS = [c for c in [
    'card_tx_count','card_fraud_count','card_fraud_rate',
    'payer_email_tx_count','payer_email_fraud_count','payer_email_fraud_rate',
    'recip_email_tx_count','recip_email_fraud_count','recip_email_fraud_rate',
    'billing_tx_count','billing_fraud_count','billing_fraud_rate',
    'device_tx_count','device_fraud_count','device_fraud_rate',
    'os_browser_tx_count','os_browser_fraud_count','os_browser_fraud_rate',
    'is_proxy','proxy_fraud_rate','prev_card_is_fraud','prev_card_dt_gap',
] if c in df.columns]

EMB_COLS  = [f'emb_{i}' for i in range(64) if f'emb_{i}' in df.columns]
EXCLUDE   = {'TransactionID','TransactionDT','isFraud'}
TAB_COLS  = [c for c in df.columns if c not in EXCLUDE and c not in GRAPH_COLS and c not in EMB_COLS]
ALL_COLS  = TAB_COLS + GRAPH_COLS + EMB_COLS

SPLIT_DT  = 12_528_000
train = df[df['TransactionDT'] <= SPLIT_DT].copy()
val   = df[df['TransactionDT'] >  SPLIT_DT].copy()
SCALE_POS = (train['isFraud'] == 0).sum() / (train['isFraud'] == 1).sum()

print(f'Train: {len(train):,}  Val: {len(val):,}')
print(f'Tabular: {len(TAB_COLS)}  Graph: {len(GRAPH_COLS)}  Embeddings: {len(EMB_COLS)}')

---
## Helper: evaluate model and log to Vertex AI Experiments

In [ ]:
def evaluate_and_log(run_name, clf, X_val, y_val, feature_cols, params):
    """Train eval + log everything to a Vertex AI Experiment run."""
    prob  = clf.predict_proba(X_val)[:, 1]
    roc   = roc_auc_score(y_val, prob)
    pr    = average_precision_score(y_val, prob)

    best_f1, best_t = 0, 0.5
    for t in np.arange(0.05, 0.95, 0.01):
        s = f1_score(y_val, prob >= t, zero_division=0)
        if s > best_f1: best_f1, best_t = s, t
    pred = (prob >= best_t).astype(int)
    f1   = f1_score(y_val, pred)
    prec = precision_score(y_val, pred, zero_division=0)
    rec  = recall_score(y_val, pred)

    metrics = dict(roc_auc=round(roc,4), pr_auc=round(pr,4),
                   f1=round(f1,4), precision=round(prec,4), recall=round(rec,4),
                   best_threshold=round(best_t,2),
                   best_iteration=int(clf.best_iteration_))

    # ── Log to Vertex AI Experiments ──────────────────────────────────────────
    # Resume existing run if it exists, otherwise create new
    try:
        aiplatform.start_run(run_name, resume=True)
    except Exception:
        aiplatform.start_run(run_name, resume=False)
    aiplatform.log_params(params)
    aiplatform.log_metrics(metrics)

    # ── Save plots to GCS and log as metadata ─────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(run_name, fontsize=13)

    # Confusion matrix
    ConfusionMatrixDisplay(confusion_matrix(y_val, pred),
                           display_labels=['Legit','Fraud']).plot(
        ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title(f'Confusion Matrix  (threshold={best_t:.2f})')

    # ROC curve
    RocCurveDisplay.from_predictions(y_val, prob, ax=axes[1], color='#e74c3c')
    axes[1].set_title(f'ROC Curve  (AUC={roc:.4f})')
    axes[1].plot([0,1],[0,1],'--', color='grey')

    # PR curve
    PrecisionRecallDisplay.from_predictions(y_val, prob, ax=axes[2], color='#3498db')
    axes[2].set_title(f'PR Curve  (AUC={pr:.4f})')

    plt.tight_layout()

    local_plot = f'{run_name}_eval.png'
    fig.savefig(local_plot, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

    gcs_plot = f'{ARTIFACTS_PATH}/{run_name}_eval.png'
    import subprocess
    subprocess.run(['gsutil','cp', local_plot, gcs_plot], capture_output=True)
    aiplatform.log_metrics({'eval_plot_gcs': 0})  # placeholder metric to keep run visible

    # Feature importance
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    fi = (pd.DataFrame({'feature': clf.feature_name_, 'importance': clf.feature_importances_})
          .sort_values('importance', ascending=False).head(25))
    color_map = {'Graph':'#e74c3c','Embedding':'#f39c12','Tabular':'#3498db'}
    def ftype(n): return 'Graph' if n in GRAPH_COLS else ('Embedding' if n in EMB_COLS else 'Tabular')
    ax2.barh(fi['feature'], fi['importance'], color=fi['feature'].map(ftype).map(color_map).tolist())
    ax2.invert_yaxis(); ax2.set_title(f'Top 25 Feature Importances - {run_name}')
    ax2.legend(handles=[Patch(facecolor=c, label=t) for t,c in color_map.items()], loc='lower right')
    plt.tight_layout()

    local_fi = f'{run_name}_importance.png'
    fig2.savefig(local_fi, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    subprocess.run(['gsutil','cp', local_fi, f'{ARTIFACTS_PATH}/{run_name}_importance.png'], capture_output=True)

    aiplatform.end_run()

    print(f'\n{run_name} logged to Vertex AI Experiments:')
    for k,v in metrics.items():
        print(f'  {k:20s}: {v}')
    print(f'  Plots saved: {ARTIFACTS_PATH}/{run_name}_*.png')

    return metrics, pred, prob

---
## Experiment 1 - Tabular Baseline

In [ ]:
LGBM = dict(n_estimators=500, learning_rate=0.05, num_leaves=63,
            scale_pos_weight=SCALE_POS, random_state=42, n_jobs=-1, verbose=-1)

print('Training Experiment 1: Tabular Baseline ...')
X_tr  = train[TAB_COLS];  y_tr  = train['isFraud']
X_val = val[TAB_COLS];    y_val = val['isFraud']

clf_base = lgb.LGBMClassifier(**LGBM)
clf_base.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
             callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(50)])

metrics_base, pred_base, prob_base = evaluate_and_log(
    run_name   = 'exp1-tabular-baseline',
    clf        = clf_base,
    X_val      = X_val,
    y_val      = y_val,
    feature_cols = TAB_COLS,
    params     = dict(
        feature_set      = 'tabular_only',
        n_features       = len(TAB_COLS),
        n_train          = len(train),
        n_val            = len(val),
        lgbm_n_estimators= 500,
        lgbm_learning_rate= 0.05,
        lgbm_num_leaves  = 63,
    )
)

---
## Experiment 2 - Graph-Enhanced (Tabular + Graph Features + FastRP Embeddings)

In [ ]:
print('Training Experiment 2: Graph-Enhanced ...')
X_tr_g  = train[ALL_COLS].fillna(-1);  y_tr_g  = train['isFraud']
X_val_g = val[ALL_COLS].fillna(-1);    y_val_g = val['isFraud']

clf_graph = lgb.LGBMClassifier(**LGBM)
clf_graph.fit(X_tr_g, y_tr_g, eval_set=[(X_val_g, y_val_g)],
              callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(50)])

metrics_graph, pred_graph, prob_graph = evaluate_and_log(
    run_name   = 'exp2-graph-enhanced',
    clf        = clf_graph,
    X_val      = X_val_g,
    y_val      = y_val_g,
    feature_cols = ALL_COLS,
    params     = dict(
        feature_set       = 'tabular_graph_embeddings',
        n_features        = len(ALL_COLS),
        n_graph_features  = len(GRAPH_COLS),
        n_embeddings      = len(EMB_COLS),
        n_train           = len(train),
        n_val             = len(val),
        lgbm_n_estimators = 500,
        lgbm_learning_rate= 0.05,
        lgbm_num_leaves   = 63,
    )
)

---
## Summary Comparison

In [ ]:
metrics_names = ['roc_auc','pr_auc','f1','precision','recall']
results = pd.DataFrame({
    'Metric':           metrics_names,
    'Tabular Baseline': [metrics_base[m]  for m in metrics_names],
    'Graph-Enhanced':   [metrics_graph[m] for m in metrics_names],
})
results['Delta'] = results['Graph-Enhanced'] - results['Tabular Baseline']
results['Lift']  = (results['Delta'] / results['Tabular Baseline'] * 100).map('{:+.1f}%'.format)
print(results.round(4).to_string(index=False))
print(f'\nView full comparison in Vertex AI console:')
print(f'https://console.cloud.google.com/vertex-ai/experiments/experiments/{EXPERIMENT_NAME}?project={PROJECT_ID}')